In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import matplotlib.pyplot as plt
import scipy as scp
import zarr
import s3fs
import trackio
from pathlib import Path
import dask.array as da
from tqdm.dask import TqdmCallback
from tqdm import tqdm

In [2]:
S3_BUCKET = "simulations"
STORE_PATH = "flow_export/A1"
fs = s3fs.S3FileSystem(profile="default", client_kwargs={'endpoint_url':'http://localhost:9000'})

ds_link = zarr.open(
    store=s3fs.S3Map(root=f"{S3_BUCKET}/{STORE_PATH}/data.zarr", s3=fs),
    mode="r",
)
channel_link = zarr.open(
    store=s3fs.S3Map(root=f"{S3_BUCKET}/{STORE_PATH}/channel.zarr", s3=fs),
    mode="r",
)
dd = da.from_zarr(ds_link)


In [3]:
dd

dask.array<from-zarr, shape=(1694, 5, 128, 129, 128), dtype=float32, chunksize=(1, 5, 128, 129, 128), chunktype=numpy.ndarray>

In [4]:
lr = 1e-3
optim = torch.optim.Adam
batch_size = 4
epochs = 20
full_slicing = np.s_[60:-10, :]
normalization_axes = (0, 2, 4,)
device = "cuda"
project_name = "channel1" 
mse_loss = nn.MSELoss(reduction="mean")
num_workers = 16

In [5]:
train_pc = 80
val_pc = 10
test_pc = 10
split_total = train_pc + val_pc + test_pc
# ignored indices between sets
spacing = 10
start_offset = 60

T = dd.shape[0]
available_T = (T - spacing * 2 - start_offset)
train_size = (available_T * train_pc) // split_total
val_size = (available_T * val_pc) // split_total
test_size = (available_T * test_pc) // split_total
val_range = range(start_offset, start_offset + val_size, 1)
train_range = range(val_size + spacing + start_offset, train_size + spacing + start_offset, 4)
print("Val slice: ", val_range)
print("Train slice: ", train_range)

Val slice:  range(60, 221)
Train slice:  range(231, 1361, 4)


In [6]:
ymeans_file = "ymeans.npy"
ystds_file = "ystds.npy"

def obtain_data(file_src, compute):
    path = Path(file_src)
    if path.exists():
        result = np.load(path)
    else:
        result = compute()
        np.save(path, result)
    return result

def compute_ymeans():
    with TqdmCallback(desc="Computing mean"):
        result = dd[full_slicing].mean(normalization_axes, keepdims=True).compute()
    return result

def compute_ystds():
    with TqdmCallback(desc="Computing std"):
        result = dd[full_slicing].std(normalization_axes, keepdims=True).compute() + 1e-10
    return result

ymeans = obtain_data(ymeans_file, compute_ymeans)
ystds = obtain_data(ystds_file, compute_ystds)
ymeans_gpu = torch.from_numpy(ymeans).to(device)
ystds_gpu = torch.from_numpy(ystds).to(device)

In [7]:
class DS(Dataset):
    def __init__(self, idx):
        self.idx = idx
        self.z = None
    def __len__(self):
        return len(self.idx)
        
    def __getitem__(self, i):
        if self.z is None:
            self.fs = s3fs.S3FileSystem(profile="default", client_kwargs={'endpoint_url':'http://localhost:9000'})
            self.z = zarr.open(
                store=s3fs.S3Map(root=f"{S3_BUCKET}/{STORE_PATH}/data.zarr", s3=self.fs),
                mode="r",
            )
        return (
            torch.from_numpy(self.z[self.idx[i], 4:5]), 
            torch.from_numpy(self.z[self.idx[i], :3]),
        )

train_ds = DS(train_range)
val_ds = DS(val_range)

train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, persistent_workers=True, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, persistent_workers=True, pin_memory=True, drop_last=True)


In [8]:
def stats(dl):
    sums = [0, 0]
    sums_sq = [0, 0]
    counts = [0, 0]

    for batch in tqdm(dl):
        for i, t in enumerate(batch):
            t = t.double()
            dims = (0, *range(2, t.ndim))
            sums[i] += t.sum(dim=dims)
            sums_sq[i] += t.square().sum(dim=dims)
            counts[i] += t.numel() // t.shape[1]

    means = [s / n for s, n in zip(sums, counts)]
    stds = [
        (ss / n - m.square()).clamp_min(0).sqrt()
        for ss, n, m in zip(sums_sq, counts, means)
    ]
    return list(zip(means, stds))

# print("Train:", stats(train_dl))
# print("Val:  ", stats(val_dl))

In [9]:
def normalized_batches(ds):
     for x, y in tqdm(ds):
        x = x.to("cuda", non_blocking=True)
        y = y.to("cuda", non_blocking=True)
        # gpu in place normalization
        x.sub_(ymeans_gpu[:, 4:5]).div_(ystds_gpu[:, 4:5])
        y.sub_(ymeans_gpu[:, :3]).div_(ystds_gpu[:, :3])
        yield x, y

def train(model, opti, loss_fn, ds):
    model.train()
    tot = 0.0
    for x, y in normalized_batches(ds):
        opti.zero_grad()
        l = loss_fn(model(x), y)
        l.backward()
        opti.step()
        tot += l.item()
    return tot / len(ds)


@torch.no_grad()
def evaluate(model, ds, loss_fn):
    model.eval()
    tot = 0.0
    for x, y in normalized_batches(ds):
        tot += loss_fn(model(x), y).item()
    return tot / len(ds)

def train_loop(model, loss_fn, ds_train, ds_val, epochs, optim, lr, project_name, model_config):
    opti = optim(model.parameters(), lr=lr)
    cfg = {"epochs": epochs, "learning_rate": lr} | model_config
    trackio.init(project=project_name, config=cfg, embed=False)

    for epoch in range(epochs):
        tr = train(model, opti, loss_fn, ds_train)
        val_mse = evaluate(model, ds_val, loss_fn)
        trackio.log({
            "epoch": epoch,
            "train/mse": tr,
            "val/mse": val_mse,
            "val/EV": 1.0 - val_mse,
        })
    trackio.finish()

In [10]:
class ConvBlock3D(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(ch, ch, 3, padding="same"),
            nn.SiLU(),
            nn.Conv3d(ch, ch, 3, padding="same"),
            nn.SiLU(),
        )
    def forward(self, x):
        return self.block(x)

class DataNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.dy = 129
        self.downsample = nn.Sequential(
            nn.Conv3d(1, 32, kernel_size=3, stride=3),
            nn.SiLU(),
            nn.Conv3d(32, 64, kernel_size=3, stride=3),
            nn.SiLU(),
        )

        self.core = nn.Sequential(
            nn.Conv3d(1, 64, 3, padding="same"),
            ConvBlock3D(64),
            ConvBlock3D(64),
            ConvBlock3D(64),
            nn.Conv3d(64, 3, 3, padding="same"),
        )

        self.upsample = nn.Sequential(
            nn.Upsample(scale_factor=3, mode="trilinear", align_corners=False),
            nn.Conv3d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=3, mode="trilinear", align_corners=False),
            nn.Conv3d(64, 3, kernel_size=3, padding=1),
            nn.ReLU(),
        )

    def forward(self, p):
        x = p.expand(-1, -1, -1, self.dy, -1)
        target_size = x.shape[2:]
        # x = self.downsample(x)
        x = self.core(x)
        # x = self.upsample(x)

        # x = F.interpolate(
        #     x, size=target_size,
        #     mode="trilinear", align_corners=False
        # )
        
        return x

In [11]:
model = DataNet()
model.to(device)
model_config = {"conv3d": "default"}

In [ ]:
train_loop(model, mse_loss, train_dl, val_dl, epochs, optim, lr, project_name, model_config)

* Trackio project initialized: channel1
* Trackio metrics logged to: /home/chancrin/.cache/huggingface/trackio
* View dashboard by running in your terminal:
trackio show --project "channel1"
* or by running in Python: trackio.show(project="channel1")
* psutil detected, enabling automatic CPU/system metrics logging
* Created new run: nice-star-11


  0%|                                        | 0/40 [00:00<?, ?it/s]

In [ ]:
# run 9 : :1
# run 10: :2
# run 11: :4
# run 12: :8